# Microsoft Fabric Public Pricelist Extractor

Fetches all Microsoft Fabric SKUs from the **Azure Retail Prices API** (no auth required)  
and surfaces them as a clean pandas DataFrame — optionally persisted to a Fabric Lakehouse Delta table.

**API reference:** https://learn.microsoft.com/en-us/rest/api/cost-management/retail-prices/azure-retail-prices

---
| Field | Value |
|---|---|
| Service filter | `Microsoft Fabric` |
| Currency | USD (configurable) |
| Auth required | No |
| Typical row count | ~1,085 (grows as new SKUs launch) |

## 1 — Imports & lightweight dependency check

In [ ]:
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    """Install *pkg* if *import_name* (or *pkg*) is not importable."""
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("requests")
_ensure("pandas")

import json
import requests
import pandas as pd
from datetime import datetime, timezone

print(f"pandas {pd.__version__} | requests {requests.__version__}")

## 2 — Configuration

In [ ]:
# ── API settings ─────────────────────────────────────────────────────────────
BASE_URL        = "https://prices.azure.com/api/retail/prices"
API_VERSION     = "2023-01-01-preview"   # latest stable version
CURRENCY_CODE   = "USD"                  # change to e.g. "EUR", "GBP" if needed

# OData filter — keep serviceName fixed; add extra clauses if you want to
# narrow to a specific region, e.g.:
#   EXTRA_FILTER = " and armRegionName eq 'eastus'"
EXTRA_FILTER    = ""

# ── Lakehouse persistence (Microsoft Fabric only) ─────────────────────────────
# Set to True to write results to a Delta table in your default Lakehouse.
SAVE_TO_LAKEHOUSE   = False
LAKEHOUSE_TABLE     = "fabric_pricelist"   # table name inside the Lakehouse
WRITE_MODE          = "overwrite"          # "overwrite" or "append"

# ── Display options ───────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",    50)
pd.set_option("display.float_format", "{:,.4f}".format)

## 3 — Fetch pricing data from the Azure Retail Prices API

In [ ]:
def fetch_fabric_prices(
    base_url: str = BASE_URL,
    currency: str = CURRENCY_CODE,
    extra_filter: str = EXTRA_FILTER,
) -> list[dict]:
    """
    Page through the Azure Retail Prices API and return all
    Microsoft Fabric items as a flat list of dicts.

    The API returns up to 100 items per page and provides a
    'NextPageLink' when more pages exist.  As of 2025 the full
    Fabric catalogue fits in a single page, but this function
    handles pagination defensively.
    """
    odata_filter = f"serviceName eq 'Microsoft Fabric'{extra_filter}"

    params = {
        "api-version":    API_VERSION,
        "currencyCode":   currency,
        "$filter":        odata_filter,
    }

    items: list[dict] = []
    url: str | None   = base_url
    page              = 0

    while url:
        page += 1
        resp = requests.get(
            url,
            params=params if page == 1 else None,   # params already encoded in NextPageLink
            timeout=30,
        )
        resp.raise_for_status()
        body = resp.json()

        batch = body.get("Items", [])
        items.extend(batch)
        print(f"  page {page}: +{len(batch):,} items  (running total: {len(items):,})")

        url = body.get("NextPageLink")   # None → loop ends

    return items


print("Fetching Microsoft Fabric prices …")
raw_items = fetch_fabric_prices()
print(f"\nDone — {len(raw_items):,} items fetched at {datetime.now(timezone.utc).isoformat()}")

## 4 — Build & clean the DataFrame

In [ ]:
# Columns returned by the API (all kept; rename a handful for clarity)
RENAME_MAP = {
    "currencyCode":          "currency",
    "armRegionName":         "region",
    "location":              "region_display",
    "retailPrice":           "retail_price",
    "unitPrice":             "unit_price",
    "unitOfMeasure":         "unit_of_measure",
    "productName":           "product_name",
    "skuName":               "sku_name",
    "meterName":             "meter_name",
    "serviceName":           "service_name",
    "serviceFamily":         "service_family",
    "serviceId":             "service_id",
    "productId":             "product_id",
    "skuId":                 "sku_id",
    "meterId":               "meter_id",
    "effectiveStartDate":    "effective_start_date",
    "tierMinimumUnits":      "tier_minimum_units",
    "isPrimaryMeterRegion":  "is_primary_meter_region",
    "armSkuName":            "arm_sku_name",
    "type":                  "price_type",
    "reservationTerm":       "reservation_term",
}

df = (
    pd.DataFrame(raw_items)
      .rename(columns=RENAME_MAP)
)

# Parse dates
df["effective_start_date"] = pd.to_datetime(df["effective_start_date"], utc=True)

# Add a snapshot timestamp so you can track when the data was pulled
df["extracted_at"] = datetime.now(timezone.utc).isoformat()

# Friendly sort
df = df.sort_values(["product_name", "sku_name", "region"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")

## 5 — Explore the data

In [ ]:
# ── Overview ──────────────────────────────────────────────────────────────────
print("Products:")
print(df["product_name"].value_counts().to_string())

print("\nPrice types:")
print(df["price_type"].value_counts().to_string())

print("\nUnits of measure:")
print(df["unit_of_measure"].value_counts().to_string())

print(f"\nRegions covered: {df['region'].nunique()}")

In [ ]:
# ── Sample rows ───────────────────────────────────────────────────────────────
DISPLAY_COLS = [
    "product_name", "sku_name", "meter_name",
    "region", "region_display",
    "price_type", "unit_of_measure",
    "retail_price", "currency",
    "effective_start_date", "reservation_term",
]

# Only keep columns that actually exist (reservation_term absent for Consumption rows)
show_cols = [c for c in DISPLAY_COLS if c in df.columns]

df[show_cols].head(10)

## 6 — Focused views

In [ ]:
# ── 6a: Fabric Capacity — Consumption prices by SKU (East US as reference) ───
capacity_eastus = (
    df[
        (df["product_name"] == "Fabric Capacity")
        & (df["price_type"]   == "Consumption")
        & (df["region"]        == "eastus")
    ]
    [["sku_name", "meter_name", "retail_price", "unit_of_measure"]]
    .sort_values("retail_price")
    .reset_index(drop=True)
)

print("Fabric Capacity — Consumption prices in East US")
capacity_eastus

In [ ]:
# ── 6b: OneLake — storage & operation prices (East US) ───────────────────────
onelake_eastus = (
    df[
        (df["product_name"] == "OneLake")
        & (df["region"]      == "eastus")
    ]
    [["sku_name", "meter_name", "retail_price", "unit_of_measure", "price_type"]]
    .sort_values("retail_price")
    .reset_index(drop=True)
)

print("OneLake — prices in East US")
onelake_eastus

In [ ]:
# ── 6c: Reservation prices for Fabric Capacity ────────────────────────────────
reservations = (
    df[
        (df["price_type"] == "Reservation")
        & (df["region"]    == "eastus")
    ]
    [["product_name", "sku_name", "reservation_term", "retail_price", "unit_of_measure"]]
    .sort_values(["product_name", "sku_name", "reservation_term"])
    .reset_index(drop=True)
)

print("Reservation prices — East US")
reservations

In [ ]:
# ── 6d: Regional price comparison for a single SKU ───────────────────────────
COMPARE_SKU = "Spark Memory Optimized Capacity Usage"   # change as needed

regional = (
    df[
        (df["sku_name"]    == COMPARE_SKU)
        & (df["price_type"] == "Consumption")
        & (df["is_primary_meter_region"] == True)
    ]
    [["region_display", "region", "retail_price", "currency"]]
    .sort_values("retail_price")
    .reset_index(drop=True)
)

print(f"Regional prices — '{COMPARE_SKU}'")
regional

In [ ]:
# ── 6e: Pivot — top 10 SKUs × top 5 regions (retail price) ───────────────────
top_skus    = df[df["price_type"] == "Consumption"]["sku_name"].value_counts().head(10).index
top_regions = ["eastus", "westeurope", "southeastasia", "australiaeast", "brazilsouth"]

pivot = (
    df[
        df["sku_name"].isin(top_skus)
        & df["region"].isin(top_regions)
        & (df["price_type"] == "Consumption")
    ]
    .pivot_table(
        index="sku_name",
        columns="region",
        values="retail_price",
        aggfunc="min",
    )
    .round(4)
)

print("Price pivot (USD) — top SKUs × selected regions")
pivot

## 7 — (Optional) Save to Fabric Lakehouse as a Delta table

Set `SAVE_TO_LAKEHOUSE = True` in **cell 2** to enable this step.  
The cell uses `spark` and `notebookutils`, which are pre-injected in the Fabric runtime.

In [ ]:
if SAVE_TO_LAKEHOUSE:
    try:
        # spark and notebookutils are available in the Fabric runtime
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()

        # Convert pandas → Spark; cast datetime to string for Delta compatibility
        df_to_save = df.copy()
        df_to_save["effective_start_date"] = df_to_save["effective_start_date"].astype(str)

        sdf = spark.createDataFrame(df_to_save)

        (
            sdf.write
               .format("delta")
               .mode(WRITE_MODE)
               .option("overwriteSchema", "true")
               .saveAsTable(LAKEHOUSE_TABLE)
        )

        print(f"Saved {len(df):,} rows to Delta table '{LAKEHOUSE_TABLE}' (mode={WRITE_MODE})")

    except ImportError:
        print(
            "PySpark not available — are you running this outside Microsoft Fabric?\n"
            "Set SAVE_TO_LAKEHOUSE = False to skip this step."
        )
else:
    print("SAVE_TO_LAKEHOUSE is False — skipping Delta write.")

## 8 — Export to CSV / Parquet (local / ADLS)

In [ ]:
# Uncomment the format you want:

# -- CSV (works anywhere) -----------------------------------------------------
# snapshot_date = datetime.now(timezone.utc).strftime("%Y%m%d")
# out_path = f"fabric_pricelist_{snapshot_date}.csv"
# df.to_csv(out_path, index=False)
# print(f"Saved → {out_path}")

# -- Parquet (efficient for downstream analytics) -----------------------------
# df.to_parquet(f"fabric_pricelist_{snapshot_date}.parquet", index=False)

# -- Fabric Files section (OneLake path) -------------------------------------
# df.to_csv("/lakehouse/default/Files/fabric_pricelist.csv", index=False)

print("Export cell ready — uncomment the desired format above.")